# CS781 capstone — MASTER notebook (all four architectures, Phases 3–7)

**How to use:** edit only the **CONTROL** cell (architecture, phase, technique, run mode, your hyperparameters), then Run All / commit.
Every phase section starts with `if PHASE == ...` so only the chosen phase runs.

**Who wrote what (for the LLM declaration):**
- *Your code, copied verbatim from your notebooks:* tokenizer (`a1_tokenizer.py`, `load_tokenizer`), the full Phase-1 code of arch 2 / 3 / 4 (one module each), your Phase-2 mask-aware arch-4 code, `load_encoder`, `load_encoder_a4`, `LoRALinear`, `LoRMALinear`, `inject`, `freeze_all_but`, `trainable_report`, and `backbone_states` (= the body of your Phase-2 `Tagger.forward`).
- *Plumbing written by Claude:* control cell layout, data download/loading/statistics, sentence packing, checkpoint finding, training/eval harness, macro-F1, submission writer, teacher-logit caching.
- *YOUR slots (marked `YOUR CODE`):* arch-1 classes (file not received yet), the Phase-3 model (pooling + document layer + head), PEFT techniques other than LoRA/LoRMA, the KD student and KD loss, and every hyperparameter.

## 0. CONTROL — the only cell you edit per run

In [1]:
ARCH      = "arch3"     # "arch1" | "arch2" | "arch3" | "arch4"
PHASE     = 4          # 3 | 4 | 5 | 6 | 7
TECHNIQUE = None      # YOUR choice. Phase 4 list: full, partial, freezing, bitfit, lorma (compulsory), adalora, adapters, kd (compulsory)
                      #   ready now: "full", "lorma", "kd"; others need your entry in EXTRA_TECHNIQUES
RUN_MODE  = "dev"       # "dev": train on train, select on dev | "final": train on train+dev, predict test, write submission.zip
BIDIR     = True        # True = bidirectional (MLM checkpoints) | False = causal (arch1, or original LM checkpoints)
SEED      = 0

# checkpoint file names, searched anywhere under /kaggle/input
CKPTS = {
    "arch1": {"lm": "phase1_arch1_ckpt.pt"},
    "arch2": {"lm": "arch2e2_ckpt.pt", "mlm": "arch2_mlm_best.pt"},
    "arch3": {"lm": "arch3_ckpt.pt",   "mlm": "/kaggle/input/datasets/ashwaryrathore/arch3-mlm-best-pt-file/arch3_mlm_best.pt"},
    "arch4": {"lm": "arch4_ckpt.pt",   "mlm": "arch4_mlm_best.pt"},
}
CKPT_KIND = "mlm" if BIDIR else "lm"      # Phase 6 (generation) must use "lm"

# ---- YOUR hyperparameters: every None must be filled before a training run ----
HP = dict(
    DEDUP=True, CHUNK_LEN=1024, OVERLAP=0.2, MAX_CHUNKS=8, KEEP="both", GRAD_CKPT=True,   # Phase-4 input (KEEP: "first" | "last" | "both")
    TRAIN_FRAC=None, TIME_BUDGET_H=None, EVAL_EVERY=None,   # Phase-4 run: fraction of train cases (None = all) | training-hours limit | dev check every N cases (None = end of epoch only)
    R=8, ALPHA=8, TARGETS=("q_proj", "k_proj", "v_proj", "out_proj"),   # technique
    MAX_SENT_TOK=166, PACK_W=1024, SENT_SEP_EOS=True, MAX_TRAIN_WIN=4,     # Phase-3 input
    EPOCHS=1, LR_ENC=None, LR_HEAD=1e-4, WARMUP_FRAC=0.1, WD=0.01,          # optimisation
    ACCUM_DOCS=None, CLASS_W_POWER=0.2,   # Phase 4: ONE label per case -> ACCUM_DOCS = cases per optimizer step (your value)
    FINAL_EPOCHS=None,                       # RUN_MODE="final": fixed epochs (no dev split to select on)
    STAGE2_EPOCHS=60, STAGE2_LR=2e-5,      # 3.6 stage 2 (doc layer on full documents) -- your values
    FOCAL_GAMMA=2,                        # used only when LOSS = "focal"
    STUDENT_LAYERS=2, KD_T=3, KD_ALPHA=0.3,   # KD runs only
)
TEACHER_TAG = None   # KD runs: tag of the teacher run whose logits are attached, e.g. "arch3_full"
CACHE_TEACHER_LOGITS = False  # True ONLY on the teacher's run (Phase 4: arch3 account): saves its logits on the training cases for KD (~2.5 h extra)
LOSS = "ce"          # "ce" = weighted cross-entropy | "focal" = your focal_loss (cell after 3.4)

def need(*keys):
    missing = [k for k in keys if HP.get(k) is None]
    assert not missing, f"fill these in the CONTROL cell first: {missing}"

RUN_TAG = f"p{PHASE}_{ARCH}_{TECHNIQUE}_{RUN_MODE}"
print("RUN:", RUN_TAG, "| checkpoint kind:", CKPT_KIND)

RUN: p4_arch3_lora_dev | checkpoint kind: mlm


## 1. Setup

In [2]:
import os; os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less fragmentation on long cases
import os, json, glob, math, time, random, zipfile, collections, re, bisect
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict

device = "cuda" if torch.cuda.is_available() else "cpu"
DEV_TYPE = "cuda" if device == "cuda" else "cpu"
EOS_ID, MASK_ID = 65354, 65355
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", device)

def find_input(name):
    if name.startswith("/") and os.path.exists(name): return name
    hits = [os.path.join(r, n) for r, ds, fs in os.walk("/kaggle/input", followlinks=True)
            for n in fs + ds if n == name]
    assert hits, f"{name} not found under /kaggle/input -- attach the dataset/notebook output that holds it (restart the session if you attached it after starting)"
    return sorted(hits, key=os.path.isdir)[0]          # prefer a real file over a folder

device: cuda


In [3]:
# official data, per phase (files as listed in the capstone PDF)
REPO = "Exploration-Lab/CS-781-Capstone"
PHASE_FILES = {
    3: ("Phase_3_RR",   ["Train/CL_train.json", "Train/CL_dev.json", "Train/IT_train.json", "Train/IT_dev.json",
                         "Train/label_vocab.json", "Test/CL_test.json", "Test/IT_test.json"]),
    4: ("Phase_4_CJPE", ["Train/train.json", "Train/dev.json", "Test/test.json"]),
    5: ("Phase_5_LSI",  ["Train/train.json", "Train/dev.json", "Test/test.json"]),
    6: ("Phase_6_Summ", ["Train/train.json", "Test/test.json"]),
    7: ("Phase_7_PCR",  ["Train/train.json", "Train/train_candidates.json", "Dev/dev.json",
                         "Dev/dev_candidates.json", "Test/test.json", "Test/test_candidates.json"]),
}

def fetch_phase(phase):
    """Uses attached copies if present (offline-safe), otherwise downloads with HF_TOKEN."""
    folder, files = PHASE_FILES[phase]
    paths = {}
    for f in files:
        local = glob.glob(f"/kaggle/input/**/{os.path.basename(f)}", recursive=True)
        if local and phase == 3:                  # phase-3 file names are unique; later phases share names (train.json ...)
            paths[f] = local[0]; continue
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import hf_hub_download
        tokn = UserSecretsClient().get_secret("HF_TOKEN")
        paths[f] = hf_hub_download(REPO, f"{folder}/{f}", repo_type="dataset", token=tokn,
                                   force_download=(phase == 4))      # the Phase-4 data was replaced on Oct 20
    if phase == 4:
        from huggingface_hub import HfApi
        print("dataset revision:", HfApi().dataset_info(REPO, token=tokn).sha)   # note this in the report)
    for f, p in paths.items():
        print(f"{f:32s} {os.path.getsize(p) // 1024:>8} KB  {p}")
    return paths

P = fetch_phase(PHASE)

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

dataset revision: 93b0059be756c6c461f3a225de25aabda7d5f84a
Train/train.json                   719380 KB  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
Train/dev.json                      43909 KB  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
Test/test.json                      34441 KB  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json


## 2. Tokenizer (your Assignment-1 BPE, verbatim)

In [4]:
%%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""
import re
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train(self,text,vocab_size):
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")
             
         

def train_tokenizer(train_text: str, vocab_size: int):
    """Train your A-1 BPE tokenizer on `train_text` and return the trained tokenizer.

    Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()

    Replace this stub with your Assignment-1 tokenizer implementation/training call.
    """
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

Writing a1_tokenizer.py


In [5]:
import re
from a1_tokenizer import BPETokeniser
mp = find_input("legal_bpe_merges.json")
raw = json.load(open(mp))
print(type(raw).__name__, len(raw))

list 65098


In [6]:
def load_tokenizer(raw):
    tok = BPETokeniser()
    if isinstance(raw, dict) and "merges" in raw:
        raw = raw["merges"]
    triples = []
    if isinstance(raw, dict):                                  # {"a,b": new_id}
        for k, v in raw.items():
            a, b = map(int, re.split(r"[,\s]+", k.strip("()[] ")))
            triples.append((a, b, int(v)))
    else:                                                      # list form
        for i, m in enumerate(raw):
            triples.append((int(m[0]), int(m[1]), int(m[2]) if len(m) == 3 else 256 + i))
    for a, b, new in sorted(triples, key=lambda t: t[2]):      # rebuild in learned order
        tok.merges[(a, b)] = new
        tok.vocab[new] = tok.vocab[a] + tok.vocab[b]
    return tok

tok = load_tokenizer(raw)
print("merges:", len(tok.merges), "| vocab:", tok.vocab_size)   # expect 65,098 | 65,354

merges: 65098 | vocab: 65354


## 3. The four architectures — each one complete, in its own module
Each cell below writes **one architecture's full Phase-1 code, verbatim from your Phase-1 notebook**, into its own file
(`arch1_model.py` … `arch4_model.py`) and imports it as `A1` … `A4`. Keeping them as separate modules means the four
sets of classes never overwrite each other (all of them define `GQAttention`, `RMSNorm`, …), and each architecture can be
shown on its own. A quick parameter count (on the `meta` device, so no memory is used) checks each one against Phase 1.

### 3.1 Architecture 1 — vanilla decoder (your A3 attention, sinusoidal positions, GELU FFN)
**YOUR CODE:** paste the model classes from `phase1-train (7).ipynb` into this cell, plus `load_encoder_a1(path) -> (encoder, cfg)` and `states_a1(encoder, idx) -> (B, T, d_model)` (hidden states after the final norm).

In [134]:
%%writefile arch1_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

# ===== YOUR CODE: paste arch-1 classes from phase1-train (7).ipynb =====

Overwriting arch1_model.py


### 3.2 Architecture 2 — LLaMA-style decoder (RoPE, RMSNorm, SwiGLU, GQA) — from `phase1-arch2-6.ipynb`

In [135]:
%%writefile arch2_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class LlamaLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)

Overwriting arch2_model.py


### 3.3 Architecture 3 — prefix-LM on the arch-2 blocks (prefix mask, mask-aware attention) — from `phase1-arch3-240218.ipynb`

In [136]:
%%writefile arch3_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

def prefix_mask(T,prefix_lens,device=None):
    causal=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
    pos=torch.arange(T,device=device)
    in_prefix=pos[None,:]<prefix_lens[:,None]
    both=in_prefix[:,:,None] & in_prefix[:,None,:]
    return (causal[None] | both)[:, None]

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x,mask=None):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        if mask is None:
            mask=self.causal_mask[:T,:T]
        scores=scores.masked_fill(~mask,float("-inf"))
        
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx,prefix_lens=None):
        if prefix_lens is not None:
            mask=prefix_mask(idx.size(1),prefix_lens,idx.device)
        else:
            mask=None
        x=self.tok_emb(idx)
        for block in self.blocks: x=block(x,mask)
        x=self.norm_f(x)
        return self.head(x)

Overwriting arch3_model.py


### 3.4 Architecture 4 — dense + MoE hybrid (4 experts, top-2, on odd layers) — from `phase1-arch4.ipynb`

In [137]:
%%writefile arch4_model.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        scores=scores.masked_fill(~self.causal_mask[:T,:T],float("-inf"))
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P).sum()
        return out.view(B,T,d)

@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x):
        x=x+self.attn(self.norm1(x))
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        x=self.norm_f(x)
        return self.head(x)
    @property
    def aux_loss(self):
        moe=[b.ffn.aux_loss for b in self.blocks if b.is_moe]
        return sum(moe)/len(moe)

Overwriting arch4_model.py


In [138]:
import importlib, arch1_model, arch2_model, arch3_model, arch4_model
for _m in (arch1_model, arch2_model, arch3_model, arch4_model): importlib.reload(_m)
A1, A2, A3, A4 = arch1_model, arch2_model, arch3_model, arch4_model

with torch.device("meta"):          # builds the Phase-1 configs without allocating memory
    n2 = sum(p.numel() for p in A2.LlamaLM(A2.LlamaConfig(65355, 1024, 10, 16, 8, 2816)).parameters())
    n3 = sum(p.numel() for p in A3.PrefixLM(A3.LlamaConfig(65355, 1024, 10, 16, 8, 2816)).parameters())
    n4 = sum(p.numel() for p in A4.HybridLM(A4.HybridConfig(65355, 768, 16, 12, 6, 2048)).parameters())
print(f"arch2 {n2:,} (Phase 1: 184,909,824) | arch3 {n3:,} (184,909,824) | arch4 {n4:,} (191,800,320)")

arch2 184,909,824 (Phase 1: 184,909,824) | arch3 184,909,824 (184,909,824) | arch4 191,800,320 (191,800,320)


### 3.5 Fine-tuning versions (the `mask` argument you added in Phase 2)
Arch 2's and arch 4's Phase-1 attention is causal-only. In Phase 2 you ran them through the **mask-aware** blocks
(arch 3's `GQAttention`, and `HybridBlock.forward(x, mask)`), which load the same weights: the parameter names are identical,
and with no mask they compute exactly the causal Phase-1 model. The cell below is your Phase-2 arch-4 code, verbatim.

In [139]:
%%writefile arch4_ft.py
import math, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

from arch3_model import RMSNorm, rope_tables, apply_rope, prefix_mask, GQAttention, SwiGLU

# ===== arch 4: dense + MoE hybrid (classes copied from my Phase 1 notebook phase1-arch4) =====
class MoEFFN(nn.Module):
    def __init__(self,d_model,d_ff,n_experts,top_k,dropout=0.0):
        super().__init__()
        self.n_experts=n_experts
        self.top_k=top_k
        self.router=nn.Linear(d_model,n_experts,bias=False)
        self.experts=nn.ModuleList([SwiGLU(d_model,d_ff,dropout) for _ in range(n_experts)])
        self.aux_loss=torch.tensor(0.0)
    def forward(self,x):
        B,T,d=x.shape
        flat=x.reshape(-1,d)
        with torch.autocast("cuda",enabled=False):
            logits=self.router(flat.float())
        probs=F.softmax(logits,dim=-1)
        top_p,top_i=probs.topk(self.top_k,dim=-1)
        top_p=top_p/top_p.sum(-1,keepdim=True)
        out=torch.zeros_like(flat)
        for e,expert in enumerate(self.experts):
            tok,slot=(top_i==e).nonzero(as_tuple=True)
            if tok.numel()==0: continue
            y=expert(flat[tok])
            out.index_add_(0,tok,(y*top_p[tok,slot,None]).to(out.dtype))
        f=torch.bincount(top_i.flatten(),minlength=self.n_experts).float()/top_i.numel()
        P_=probs.mean(0)
        self.aux_loss=self.n_experts*(f*P_).sum()
        return out.view(B,T,d)

@dataclass
class HybridConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0
    n_experts:int=4
    top_k:int=2
    d_ff_expert:int=1024

class HybridBlock(nn.Module):
    def __init__(self,cfg,layer_id):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.is_moe=(layer_id%2==1)
        if self.is_moe:
            self.ffn=MoEFFN(cfg.d_model,cfg.d_ff_expert,cfg.n_experts,cfg.top_k,cfg.dropout)
        else:
            self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):            # mask arg added so the Phase-2 Tagger can call block(x, mask)
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class HybridLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([HybridBlock(cfg,i) for i in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx):
        x=self.tok_emb(idx)
        for block in self.blocks:
            x=block(x)
        return self.head(self.norm_f(x))

def load_encoder_a4(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = HybridConfig(**ck["cfg"])
    enc = HybridLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val", float("nan")), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params",
          "| MoE layers", sum(b.is_moe for b in enc.blocks), "of", len(enc.blocks))
    del ck
    return enc, cfg

Overwriting arch4_ft.py


## 4. MLM adaptation — kept aside
MLM training runs in its own notebooks (`arch3-mlm`, and the `arch2-mlm` copy of it), not here.
Their outputs (`arch2_mlm_best.pt`, `arch3_mlm_best.pt`, later `arch4_mlm_best.pt`) are plain checkpoints in the usual
`{"model", "cfg"}` format, so section 5 loads them exactly like the Phase-1 checkpoints.

## 5. Backbone loader

In [140]:
import torch.utils.checkpoint
import arch4_ft; importlib.reload(arch4_ft)
from arch3_model import LlamaConfig, PrefixLM, prefix_mask      # used by your load_encoder below
from arch4_ft import load_encoder_a4

def load_encoder(path):
    ck = torch.load(path, map_location="cpu")
    cfg = LlamaConfig(**ck["cfg"])
    enc = PrefixLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val", float("nan")), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

def backbone_states(enc, idx, prefix_lens=None):
    """Your Phase-2 Tagger.forward without the tag head: hidden states after norm_f.
    prefix_lens = window lengths -> fully bidirectional over the real tokens; None -> causal."""
    mask = prefix_mask(idx.size(1), prefix_lens, idx.device) if prefix_lens is not None else None
    x = enc.tok_emb(idx)
    for block in enc.blocks:
        if HP.get("GRAD_CKPT") and enc.training and torch.is_grad_enabled():
            x = torch.utils.checkpoint.checkpoint(block, x, mask, use_reentrant=False)   # memory switch: recompute in backward
        else:
            x = block(x, mask)
    return enc.norm_f(x)

def load_backbone(arch=None, kind=None):
    """arch2 / arch3 -> PrefixLM (arch-3 module), arch4 -> mask-aware HybridLM (arch4_ft), arch1 -> your arch1_model."""
    arch, kind = arch or ARCH, kind or CKPT_KIND
    assert kind in CKPTS[arch], f"{arch} has no '{kind}' checkpoint listed in CKPTS"
    path = find_input(CKPTS[arch][kind])
    if arch in ("arch2", "arch3"):
        enc, cfg = load_encoder(path)
    elif arch == "arch4":
        enc, cfg = load_encoder_a4(path)
    else:
        assert hasattr(A1, "load_encoder_a1"), "fill section 3.1 (arch 1) first"
        enc, cfg = A1.load_encoder_a1(path)
    return enc, cfg.d_model

def encoder_states(enc, win_ids, win_lens):
    """What your Phase-3 model calls: (n_win, L) ids + (n_win,) lengths -> (n_win, L, d_model)."""
    if ARCH == "arch1":
        return A1.states_a1(enc, win_ids)
    return backbone_states(enc, win_ids, win_lens if BIDIR else None)

## 6. Fine-tuning techniques
Your LoRA / LoRMA and helpers (verbatim), then `apply_technique`, then the slot for the techniques not written yet.

In [141]:
# ===== PEFT plumbing: freeze the backbone, report what is trainable =====
def freeze_all_but(model, keep):
    """keep(name, param) -> True if this parameter should train."""
    for n, p in model.named_parameters():
        p.requires_grad = bool(keep(n, p))

def trainable_report(model, tag=""):
    seen = {id(p): p for p in model.parameters()}.values()
    tot = sum(p.numel() for p in seen); tr = sum(p.numel() for p in seen if p.requires_grad)
    print(f"{tag} trainable {tr:,} / {tot:,} params ({tr / tot:.3%})")

In [142]:
# ===== LoRA layer (my implementation): W' = W + (alpha/r) B A, low-rank term on the INPUT x =====
import torch
import torch.nn as nn
import math
class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r:int,alpha:float):
        super().__init__()
        self.b=base
        for p in self.b.parameters():p.requires_grad=False
        i,o=base.in_features,base.out_features
        self.lora_A=nn.Parameter(torch.empty(r,i))
        self.lora_B=nn.Parameter(torch.zeros(o,r))
        nn.init.kaiming_uniform_(self.lora_A,a=math.sqrt(5))
        self.s=alpha/r
    def forward(self,x):
        return self.b(x)+self.s*(x@self.lora_A.T@self.lora_B.T)

In [143]:
# ===== LoRMA layer (my implementation): W' = (I + (alpha/r) B A) W, applied to the frozen Linear's output =====
import torch
import torch.nn as nn
import math

class LoRMALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base

        for param in self.base.parameters():
            param.requires_grad = False

        d_out = base.out_features

        self.lorma_A = nn.Parameter(torch.empty(r, d_out))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))

        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))

        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (h @ self.lorma_A.T @ self.lorma_B.T)

In [144]:
def inject(model, wrap_fn, targets):
    """Replace every nn.Linear child whose attribute name is in `targets` with wrap_fn(linear)."""
    n = 0
    for _, mod in list(model.named_modules()):
        for t in targets:
            child = getattr(mod, t, None)
            if isinstance(child, nn.Linear):
                setattr(mod, t, wrap_fn(child)); n += 1
    return n

In [145]:
def apply_technique(model, technique=None):
    """Wraps model.encoder for the chosen technique and freezes accordingly.
    Rule used for every PEFT method: the technique's own parameters + everything outside model.encoder
    (your pooling / document layer / head) train; the rest of the encoder is frozen."""
    t = technique or TECHNIQUE
    if t == "full":
        for p in model.parameters(): p.requires_grad = True
    elif t == "lora":
        need("R", "ALPHA")
        n = inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
        freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
        print("LoRA wrapped", n, "Linears")
    elif t == "lorma":
        need("R", "ALPHA")
        n = inject(model.encoder, lambda lin: LoRMALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
        freeze_all_but(model, lambda n_, p: ("lorma" in n_) or not n_.startswith("encoder."))
        print("LoRMA wrapped", n, "Linears")
    elif t in EXTRA_TECHNIQUES:
        EXTRA_TECHNIQUES[t](model)
    else:
        raise NotImplementedError(f"technique '{t}' not written yet -- add it to EXTRA_TECHNIQUES in the next cell")
    trainable_report(model, f"{ARCH}+{t}")
    return model

In [146]:
def apply_rslora(model):
    need("R", "ALPHA")
    inject(model.encoder, lambda lin: LoRALinear(lin, HP["R"], HP["ALPHA"]), HP["TARGETS"])
    
    count = 0
    scale = HP["ALPHA"] / (HP["R"] ** 0.5)
    
    for module in model.encoder.modules():
        if isinstance(module, LoRALinear):
            module.s = scale
            count += 1
            
    freeze_all_but(model, lambda n_, p: ("lora" in n_) or not n_.startswith("encoder."))
    print("rsLoRA wrapped", count, "Linears | scale alpha/sqrt(r) =", round(scale, 4))

EXTRA_TECHNIQUES = {
    "rslora": apply_rslora,
}

# PHASE 3 — Rhetorical Role prediction

### 3.1 Data + statistics

In [147]:
if PHASE == 3:
    def to_docs(obj):
        if isinstance(obj, list):                                            # [ {id, text, labels}, ... ]
            return obj
        vals = list(obj.values())
        if all(isinstance(v, dict) for v in vals):                           # {doc_id: {text, labels}}
            return [dict(id=k, **v) for k, v in obj.items()]
        if "text" in obj and isinstance(obj["text"], list) and obj["text"] and isinstance(obj["text"][0], list):
            n = len(obj["text"])                                             # columnar {id:[...], text:[[...]], labels:[[...]]}
            docs = [{k: obj[k][i] for k in obj if isinstance(obj[k], list) and len(obj[k]) == n} for i in range(n)]
            for i, d in enumerate(docs): d.setdefault("id", str(i))
            return docs
        for v in vals:
            if isinstance(v, list) and v and isinstance(v[0], dict):         # {"data": [ {...}, ... ]}
                return v
        if all(isinstance(v, list) and (not v or isinstance(v[0], str)) for v in vals):
            return [{"id": k, "text": v} for k, v in obj.items()]            # {doc_id: [sentences]}
        raise ValueError(f"unknown layout: top keys {list(obj)[:5]}, first value type {type(vals[0])}")

    vocab = json.load(open(P["Train/label_vocab.json"]))
    if isinstance(vocab, dict) and all(str(k).isdigit() for k in vocab):          # {"0": "Fact", ...}
        LABELS = [vocab[k] for k in sorted(vocab, key=int)]
    elif isinstance(vocab, dict):                                                  # {"Fact": 0, ...}
        LABELS = sorted(vocab, key=lambda k: vocab[k]) if all(isinstance(v, int) for v in vocab.values()) else list(vocab)
    else:
        LABELS = list(vocab)
    L2I = {l: i for i, l in enumerate(LABELS)}
    print("labels:", LABELS)

    RAW = {}
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            key = f"{'Test' if split == 'test' else 'Train'}/{dom}_{split}.json"
            obj = json.load(open(P[key]))
            try:
                RAW[(dom, split)] = to_docs(obj)
            except Exception as e:
                print(f"!! {key}: top-level type {type(obj).__name__}, "
                      f"first keys {list(obj)[:3] if isinstance(obj, dict) else '-'}, "
                      f"first item {str(obj[0] if isinstance(obj, list) else next(iter(obj.values())))[:200]}")
                raise
            d0 = RAW[(dom, split)][0]
            print(f"{dom}_{split:5s}: loaded {len(RAW[(dom, split)])} docs | keys {list(d0.keys())}")

    ids_cl = {d["id"] for s in ("train", "dev", "test") for d in RAW[("CL", s)]}
    ids_it = {d["id"] for s in ("train", "dev", "test") for d in RAW[("IT", s)]}
    test_ids = [d["id"] for dom in ("CL", "IT") for d in RAW[(dom, "test")]]
    assert len(test_ids) == len(set(test_ids)), "duplicate test ids across CL/IT -- the output dict would collide"
    print("CL/IT id overlap:", len(ids_cl & ids_it))

    def pct(xs, q):
        xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
    for (dom, split), docs in RAW.items():
        ns = [len(d["text"]) for d in docs]
        print(f"{dom}_{split:5s}: {len(docs):4d} docs | {sum(ns):6d} sentences | sent/doc p50 {pct(ns, .5)} max {max(ns)}")
        if "labels" in docs[0]:
            assert all(len(d["labels"]) == len(d["text"]) for d in docs), "label/sentence length mismatch"
            assert not ({l for d in docs for l in d["labels"]} - set(LABELS)), "unknown label in data"
    for dom in ("CL", "IT"):
        c = collections.Counter(l for s in ("train", "dev") for d in RAW[(dom, s)] for l in d["labels"])
        tot = sum(c.values())
        print(f"\n{dom} train+dev label share:", {l: f"{100 * c[l] / tot:.1f}%" for l in LABELS})

labels: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
CL_train: loaded 40 docs | keys ['id', 'text', 'labels']
CL_dev  : loaded 5 docs | keys ['id', 'text', 'labels']
CL_test : loaded 5 docs | keys ['id', 'text']
IT_train: loaded 40 docs | keys ['id', 'text', 'labels']
IT_dev  : loaded 5 docs | keys ['id', 'text', 'labels']
IT_test : loaded 5 docs | keys ['id', 'text']
CL/IT id overlap: 0
CL_train:   40 docs |  10593 sentences | sent/doc p50 123 max 1816
CL_dev  :    5 docs |   1396 sentences | sent/doc p50 184 max 848
CL_test :    5 docs |   1339 sentences | sent/doc p50 114 max 942
IT_train:   40 docs |   6252 sentences | sent/doc p50 129 max 442
IT_dev  :    5 docs |    746 sentences | sent/doc p50 120 max 382
IT_test :    5 docs |    858 sentences | sent/doc p50 159 max 420

CL train+dev label share:

In [148]:
def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
if PHASE == 3:
    for split in ("train", "dev", "test"):
        sl = [len(tok.encode(" " + s.strip())) for dom in ("CL", "IT") for d in RAW[(dom, split)] for s in d["text"]]
        dl = [sum(len(tok.encode(" " + s.strip())) for s in d["text"]) for dom in ("CL", "IT") for d in RAW[(dom, split)]]
        print(f"{split:5s} tokens/sentence p50 {pct(sl,.5)} p90 {pct(sl,.9)} p95 {pct(sl,.95)} p99 {pct(sl,.99)} max {max(sl)} "
              f"| tokens/doc p50 {pct(dl,.5)} p90 {pct(dl,.9)} max {max(dl)}")

train tokens/sentence p50 32 p90 72 p95 91 p99 166 max 1183 | tokens/doc p50 5169 p90 18131 max 69496
dev   tokens/sentence p50 33 p90 69 p95 89 p99 166 max 410 | tokens/doc p50 5651 p90 33898 max 33898
test  tokens/sentence p50 32 p90 75 p95 96 p99 178 max 622 | tokens/doc p50 7636 p90 36050 max 36050


### 3.2 Sentence encoding + packing into encoder windows
Every sentence keeps a span `(window, start, end)`; whole sentences are packed into windows of at most `PACK_W` tokens, so a document is encoded with neighbouring sentences as context.

In [149]:
if PHASE == 3:
    need("MAX_SENT_TOK", "PACK_W", "SENT_SEP_EOS")
    assert HP["MAX_SENT_TOK"] + 1 <= HP["PACK_W"] <= 1024

    SENT_CACHE = {}
    def sent_ids(s):
        if s not in SENT_CACHE:
            SENT_CACHE[s] = tok.encode(" " + s.strip())
        return SENT_CACHE[s]

    def pack_doc(sentences):
        windows, spans, cur, n_trunc, n_empty = [], [], [], 0, 0
        for s in sentences:
            ids = sent_ids(s)
            if not ids: ids, n_empty = [EOS_ID], n_empty + 1
            if len(ids) > HP["MAX_SENT_TOK"]: ids, n_trunc = ids[:HP["MAX_SENT_TOK"]], n_trunc + 1
            piece = ids + ([EOS_ID] if HP["SENT_SEP_EOS"] else [])
            if cur and len(cur) + len(piece) > HP["PACK_W"]:
                windows.append(cur); cur = []
            spans.append((len(windows), len(cur), len(cur) + len(ids)))
            cur = cur + piece
        if cur: windows.append(cur)
        return windows, spans, n_trunc, n_empty

    def build_items(dom, split):
        items, tr, em = [], 0, 0
        for d in RAW[(dom, split)]:
            w, sp, t_, e_ = pack_doc(d["text"]); tr += t_; em += e_
            items.append(dict(id=d["id"], domain=dom, windows=w, spans=sp,
                              labels=[L2I[l] for l in d["labels"]] if "labels" in d else None,
                              sent_idx=list(range(len(sp)))))
        return items, tr, em

    ITEMS = {}
    for dom in ("CL", "IT"):
        for split in ("train", "dev", "test"):
            ITEMS[(dom, split)], tr, em = build_items(dom, split)
            nw = [len(it["windows"]) for it in ITEMS[(dom, split)]]
            print(f"{dom}_{split:5s}: windows/doc p50 {sorted(nw)[len(nw)//2]} max {max(nw)} | "
                  f"truncated sentences {tr} | empty sentences {em}")
    TRAIN = ITEMS[("CL", "train")] + ITEMS[("IT", "train")]
    DEV   = ITEMS[("CL", "dev")]   + ITEMS[("IT", "dev")]
    TEST  = ITEMS[("CL", "test")]  + ITEMS[("IT", "test")]

    # check: spans point at exactly the (possibly truncated) sentence tokens
    it = TRAIN[0]
    ok = all(it["windows"][w][s:e] == sent_ids(it_s)[:e - s] for (w, s, e), it_s in
             zip(it["spans"], RAW[("CL", "train")][0]["text"]) if sent_ids(it_s))
    print("span check on first doc:", ok)

CL_train: windows/doc p50 6 max 71 | truncated sentences 73 | empty sentences 0
CL_dev  : windows/doc p50 8 max 35 | truncated sentences 12 | empty sentences 0
CL_test : windows/doc p50 4 max 37 | truncated sentences 12 | empty sentences 0
IT_train: windows/doc p50 6 max 21 | truncated sentences 95 | empty sentences 0
IT_dev  : windows/doc p50 5 max 16 | truncated sentences 9 | empty sentences 0
IT_test : windows/doc p50 8 max 17 | truncated sentences 13 | empty sentences 0
span check on first doc: True


### 3.3 YOUR CODE — the Phase-3 model
**Contract the harness relies on:**
- the Phase-1 backbone is stored as `self.encoder` (so the freezing rule in `apply_technique` works);
- `forward(win_ids, win_lens, spans)` with `win_ids` LongTensor `(n_win, L)` (padded with EOS), `win_lens` LongTensor `(n_win,)`, `spans` = list of `(window, start, end)` per sentence, in document order;
- returns logits `(n_sentences, len(LABELS))`, one row per span, same order;
- get token states with `encoder_states(self.encoder, win_ids, win_lens)` → `(n_win, L, d_model)`.

In [150]:
import torch
import torch.nn as nn

class RRModel(nn.Module):
    def __init__(self, encoder, d_model, n_labels, dropout=0.1, lstm_hidden=256):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        
        self.doc_lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=lstm_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        self.head = nn.Linear(2 * lstm_hidden, n_labels)

    def forward(self, win_ids, win_lens, spans):
        H = encoder_states(self.encoder, win_ids, win_lens)
        
        s_list = []
        for w_idx, start, end in spans:
            s_list.append(H[w_idx, start:end].mean(dim=0))
        S = torch.stack(s_list)
        
        S = S.unsqueeze(0)
        lstm_out, _ = self.doc_lstm(S)
        S = lstm_out.squeeze(0)
        
        S = self.dropout(S)
        logits = self.head(S)
        
        return logits

In [151]:
def build_student():
    need("STUDENT_LAYERS")
    enc, d_model = load_backbone()
    
    enc.blocks = torch.nn.ModuleList(list(enc.blocks)[:HP["STUDENT_LAYERS"]])
    
    m = RRModel(enc, d_model, len(LABELS))
    
    for param in m.parameters():
        param.requires_grad = True
        
    return m.to(device)

def kd_loss(logits, y, cw, teacher=None):
    hard_loss = torch.nn.functional.cross_entropy(logits, y, weight=cw)
    
    if teacher is None:
        return hard_loss
        
    T = HP["KD_T"]
    alpha = HP["KD_ALPHA"]
    
    student_log_probs = torch.nn.functional.log_softmax(logits / T, dim=-1)
    teacher_probs = torch.nn.functional.softmax(teacher / T, dim=-1)
    
    soft_loss = torch.nn.functional.kl_div(
        student_log_probs, 
        teacher_probs, 
        reduction='batchmean'
    ) * (T ** 2)
    
    return (1.0 - alpha) * hard_loss + alpha * soft_loss

### 3.4 Harness (training, evaluation, submission)

In [152]:
def weighted_ce(logits, y, cw, teacher=None):
    # per-item loss x weight of its own class, then mean. (F.cross_entropy(weight=cw) divides by the batch's
    # weight sum, which cancels the weight completely when a batch holds ONE item -- the Phase-4 case.)
    return (F.cross_entropy(logits, y, reduction="none") * cw[y]).mean()

def crop(item, max_win):
    """Training-time crop to <= max_win consecutive windows (whole sentences only)."""
    n = len(item["windows"])
    if max_win is None or n <= max_win:
        return item
    w0 = random.randint(0, n - max_win)
    keep = [i for i, (w, _, _) in enumerate(item["spans"]) if w0 <= w < w0 + max_win]
    return dict(item, windows=item["windows"][w0:w0 + max_win],
                spans=[(item["spans"][i][0] - w0, item["spans"][i][1], item["spans"][i][2]) for i in keep],
                labels=[item["labels"][i] for i in keep] if item["labels"] is not None else None,
                sent_idx=[item["sent_idx"][i] for i in keep])

def to_tensors(item):
    L = max(len(w) for w in item["windows"])
    x = torch.full((len(item["windows"]), L), EOS_ID, dtype=torch.long)
    for i, w in enumerate(item["windows"]):
        x[i, :len(w)] = torch.tensor(w)
    lens = torch.tensor([len(w) for w in item["windows"]])
    return x.to(device), lens.to(device)

def class_weights_rr(items, power):
    cnt = torch.zeros(len(LABELS))
    for it in items:
        cnt += torch.bincount(torch.tensor(it["labels"]), minlength=len(LABELS)).float()
    w = (cnt.sum() / cnt.clamp(min=1)) ** power
    return (w / w.mean()).to(device)

from sklearn.metrics import f1_score
def macro_f1(gold, pred):
    """Official Phase-3 metric (same as evaluate_phase3.py): macro over ALL 13 labels, zero_division=0."""
    g = [LABELS[x] for d in gold for x in d]
    p = [LABELS[x] for d in pred for x in d]
    per = f1_score(g, p, labels=LABELS, average=None, zero_division=0)
    return float(per.mean()), {l: float(v) for l, v in zip(LABELS, per)}

@torch.no_grad()
def predict_rr(model, items):
    model.eval(); out = {}
    for it in items:
        x, lens = to_tensors(it)
        with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
            lg = model(x, lens, it["spans"])
        out[it["id"]] = lg.float().cpu()
    model.train()
    return out                          # doc id -> (n_sent, n_labels) logits

def evaluate(items, logits_by_id, show_per_class=False):
    res = {}
    for dom in ("CL", "IT", "ALL"):
        sub = [it for it in items if dom == "ALL" or it["domain"] == dom]
        if not sub: continue
        f, per = macro_f1([it["labels"] for it in sub], [logits_by_id[it["id"]].argmax(-1).tolist() for it in sub])
        res[dom] = f
        if show_per_class and dom == "ALL":
            for l, v in per.items(): print(f"    {l:24s} {v:.3f}")
    return res

def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

def train_rr(model, train_items, dev_items, loss_fn=weighted_ce, teacher=None, *, EPOCHS, LR_ENC, LR_HEAD,
             WARMUP_FRAC, WD, ACCUM_DOCS, CLASS_W_POWER, MAX_TRAIN_WIN, CLIP=1.0, seed=0):
    """Per-document training with gradient accumulation. teacher: doc id -> (n_sent, n_labels) logits (KD runs).
    With dev_items: keeps the best-dev trainable weights. Without: keeps the last epoch."""
    torch.manual_seed(seed); random.seed(seed)
    train_items = list(train_items)
    cw = class_weights_rr(train_items, CLASS_W_POWER)
    enc_p  = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
    groups = [g for g in ({"params": enc_p, "lr": LR_ENC}, {"params": head_p, "lr": LR_HEAD}) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=WD)
    steps_ep = math.ceil(len(train_items) / ACCUM_DOCS); total = EPOCHS * steps_ep
    warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
    params = [p for g in groups for p in g["params"]]
    best_f, best_state, hist = -1.0, None, []
    model.train()
    for ep in range(EPOCHS):
        random.shuffle(train_items); t0 = time.time(); tot = 0.0
        for i, item in enumerate(train_items):
            it = crop(item, MAX_TRAIN_WIN)
            x, lens = to_tensors(it)
            y = torch.tensor(it["labels"], device=device)
            tch = teacher[it["id"]][it["sent_idx"]].to(device) if teacher is not None else None
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                lg = model(x, lens, it["spans"])
            loss = loss_fn(lg.float(), y, cw, teacher=tch) / ACCUM_DOCS
            scaler.scale(loss).backward(); tot += loss.item() * ACCUM_DOCS
            if (i + 1) % ACCUM_DOCS == 0 or i + 1 == len(train_items):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        msg = f"epoch {ep + 1}/{EPOCHS} | train loss {tot / len(train_items):.4f} | {time.time() - t0:.0f}s"
        if dev_items:
            res = evaluate(dev_items, predict_rr(model, dev_items))
            msg += " | dev macro-F1 " + " ".join(f"{k} {v:.4f}" for k, v in res.items())
            hist.append(res)
            if res["ALL"] > best_f:
                best_f, best_state = res["ALL"], trainable_state(model)
        print(msg)
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
        print("BEST dev macro-F1 (ALL):", round(best_f, 4))
    return best_f, hist

def write_submission(logits_by_id, tag):
    pred = {}
    for it in TEST:
        lab = [LABELS[i] for i in logits_by_id[it["id"]].argmax(-1).tolist()]
        assert len(lab) == len(it["spans"]), f"length mismatch for {it['id']}"
        pred[it["id"]] = lab
    want = {d["id"]: len(d["text"]) for dom in ("CL", "IT") for d in RAW[(dom, "test")]}
    assert set(pred) == set(want) and all(len(pred[k]) == want[k] for k in want), "ids / sentence counts do not match the test files"
    assert all(l in L2I for v in pred.values() for l in v)
    jp, zp = f"/kaggle/working/prediction_{tag}.json", f"/kaggle/working/submission_{tag}.zip"
    json.dump(pred, open(jp, "w"))
    with zipfile.ZipFile(zp, "w") as z:
        z.write(jp, arcname="prediction.json")          # must sit at the zip root
    print(f"wrote {zp} | docs {len(pred)} | sentences {sum(map(len, pred.values()))}")

In [153]:
def focal_loss(logits, y, cw, teacher=None):
    log_pt = torch.nn.functional.log_softmax(logits, dim=-1)
    log_p = log_pt.gather(1, y.unsqueeze(1)).squeeze(1)
    p = torch.exp(log_p)
    focal_factor = (1 - p) ** HP["FOCAL_GAMMA"]
    per_sentence_loss = -cw[y] * focal_factor * log_p
    return per_sentence_loss.mean()

def train_loss():
    if LOSS == "focal":
        need("FOCAL_GAMMA")
        return focal_loss
    return weighted_ce

### 3.5 Run (uses the CONTROL cell)

In [154]:
import os
for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
    for n in dirs + files:
        if "arch3" in n or "mlm" in n:
            p = os.path.join(root, n)
            print("DIR " if os.path.isdir(p) else "FILE", p, os.path.getsize(p) if os.path.isfile(p) else "")

DIR  /kaggle/input/datasets/ashwaryrathore/arch3-mlm-best-pt-file 
FILE /kaggle/input/datasets/ashwaryrathore/arch3-mlm-best-pt-file/arch3_mlm_best.pt 739680445


In [155]:
if PHASE == 3:
    need("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "CLASS_W_POWER", "MAX_TRAIN_WIN")
    settings = {k: HP[k] for k in ("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS",
                                   "CLASS_W_POWER", "MAX_TRAIN_WIN")}
    teacher, loss_fn = None, train_loss()
    if TECHNIQUE == "kd":
        assert TEACHER_TAG, "set TEACHER_TAG in the CONTROL cell and attach that run's teacher-logits file"
        teacher = torch.load(find_input(f"teacher_logits_{TEACHER_TAG}_{RUN_MODE}.pt"))
        model, loss_fn = build_student().to(device), kd_loss
        trainable_report(model, "KD student")
    else:
        model = build_rr_model()

    if RUN_MODE == "dev":
        train_items, dev_items = TRAIN, DEV
    else:
        need("FINAL_EPOCHS"); settings["EPOCHS"] = HP["FINAL_EPOCHS"]
        train_items, dev_items = TRAIN + DEV, None

    t_start = time.time()
    best_f, hist = train_rr(model, train_items, dev_items, loss_fn, teacher, seed=SEED, **settings)
    print(f"training time {(time.time() - t_start) / 60:.1f} min")
    torch.save(trainable_state(model), f"/kaggle/working/weights_{RUN_TAG}.pt")   # trainable params only

    if RUN_MODE == "dev":
        dev_logits = predict_rr(model, DEV)
        print("DEV:", evaluate(DEV, dev_logits, show_per_class=True))
        torch.save(dev_logits, f"/kaggle/working/devlogits_{RUN_TAG}.pt")          # for ensembling later
        test_logits = predict_rr(model, TEST)                                         # submit straight from the best-dev model
        torch.save(test_logits, f"/kaggle/working/testlogits_{RUN_TAG}.pt")
        write_submission(test_logits, RUN_TAG)
        json.dump(hist, open(f"/kaggle/working/hist_{RUN_TAG}.json", "w"))
    else:
        test_logits = predict_rr(model, TEST)
        torch.save(test_logits, f"/kaggle/working/testlogits_{RUN_TAG}.pt")        # for ensembling later
        write_submission(test_logits, RUN_TAG)

    if CACHE_TEACHER_LOGITS:                       # this run will serve as a KD teacher
        tl = predict_rr(model, train_items)
        torch.save(tl, f"/kaggle/working/teacher_logits_{ARCH}_{TECHNIQUE}_{RUN_MODE}.pt")
        print("saved teacher logits for", len(tl), "training docs")

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
LoRA wrapped 40 Linears
arch3+lora trainable 3,205,645 / 188,116,493 params (1.704%)
epoch 1/30 | train loss 2.4459 | 39s | dev macro-F1 CL 0.0684 IT 0.0747 ALL 0.0713
epoch 2/30 | train loss 2.0408 | 40s | dev macro-F1 CL 0.1361 IT 0.1142 ALL 0.1296
epoch 3/30 | train loss 1.7274 | 41s | dev macro-F1 CL 0.2074 IT 0.2474 ALL 0.2293
epoch 4/30 | train loss 1.3821 | 41s | dev macro-F1 CL 0.2772 IT 0.3272 ALL 0.3037
epoch 5/30 | train loss 1.1737 | 41s | dev macro-F1 CL 0.2607 IT 0.3209 ALL 0.3081
epoch 6/30 | train loss 1.1266 | 41s | dev macro-F1 CL 0.3042 IT 0.4096 ALL 0.3630
epoch 7/30 | train loss 1.0084 | 41s | dev macro-F1 CL 0.3266 IT 0.4338 ALL 0.3989
epoch 8/30 | train loss 0.9694 | 41s | dev macro-F1 CL 0.2895 IT 0.4372 ALL 0.3846
epoch 9/30 | train loss 0.9471 | 41s | dev macro-F1 CL 0.3490 IT 0.4552 ALL 0.4222
epoch 10/30 | train loss 0.8120 | 41s | dev macro-F1 CL 0.3401 IT 0.4616 ALL 0.4224
epo

### 3.6 Stage 2 — document layer on full documents
Run right after 3.5 (same session). Writes `submission_<tag>_s2.zip` plus dev/test logits for ensembling.

In [163]:
# ===== 3.6 Stage 2: train doc_lstm + head on FULL documents (plumbing; your layers, unchanged) =====
# Run right after 3.5 in the same session: it reuses `model`. Encoder + LoRA are frozen; sentence vectors are
# computed once per document (same mean-over-span pooling as RRModel.forward), then only your doc_lstm + head train,
# on whole-document sequences (no MAX_TRAIN_WIN crop).
RUN_STAGE2 = True
if PHASE == 3 and RUN_STAGE2:
    need("STAGE2_EPOCHS", "STAGE2_LR")
    assert all(hasattr(model, a) for a in ("doc_lstm", "head", "dropout")), "needs your BiLSTM RRModel"

    @torch.no_grad()
    def sentence_vectors(m, items, chunk=8):
        m.eval(); out = {}
        for it in items:
            x, lens = to_tensors(it)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                H = torch.cat([encoder_states(m.encoder, x[i:i + chunk], lens[i:i + chunk])
                               for i in range(0, x.size(0), chunk)])
            out[it["id"]] = torch.stack([H[w, s:e].float().mean(0) for w, s, e in it["spans"]]).cpu()
        return out

    def doc_logits(m, S):                         # = your RRModel.forward after pooling
        out, _ = m.doc_lstm(S.unsqueeze(0).to(device))
        return m.head(m.dropout(out.squeeze(0)))

    s2_train = list(TRAIN) if RUN_MODE == "dev" else list(TRAIN + DEV)
    SV = sentence_vectors(model, s2_train + DEV + TEST)
    print("sentence vectors cached for", len(SV), "docs")

    for p in model.parameters(): p.requires_grad = False
    doc_params = list(model.doc_lstm.parameters()) + list(model.head.parameters())
    for p in doc_params: p.requires_grad = True
    opt2 = torch.optim.AdamW(doc_params, lr=HP["STAGE2_LR"], weight_decay=HP["WD"])
    cw2 = class_weights_rr(s2_train, HP["CLASS_W_POWER"])

    @torch.no_grad()
    def s2_predict(items):
        model.eval(); out = {it["id"]: doc_logits(model, SV[it["id"]]).float().cpu() for it in items}; model.train()
        return out

    best2, state2 = -1.0, None
    for ep in range(HP["STAGE2_EPOCHS"]):
        random.shuffle(s2_train); model.train(); tot = 0.0
        for it in s2_train:
            lg = doc_logits(model, SV[it["id"]])
            loss = train_loss()(lg.float(), torch.tensor(it["labels"], device=device), cw2)
            opt2.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(doc_params, 1.0); opt2.step(); tot += loss.item()
        res = evaluate(DEV, s2_predict(DEV))
        print(f"stage2 epoch {ep + 1} | loss {tot / len(s2_train):.4f} | dev " +
              " ".join(f"{k} {v:.4f}" for k, v in res.items()) + ("" if RUN_MODE == "dev" else "  (dev is in training data)"))
        if RUN_MODE == "dev" and res["ALL"] > best2:
            best2, state2 = res["ALL"], {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    if state2 is not None:
        model.load_state_dict(state2, strict=False); print("stage2 BEST dev macro-F1:", round(best2, 4))

    print("DEV:", evaluate(DEV, s2_predict(DEV), show_per_class=True))
    tl2 = s2_predict(TEST)
    torch.save(s2_predict(DEV), f"/kaggle/working/devlogits_{RUN_TAG}_s2.pt")
    torch.save(tl2, f"/kaggle/working/testlogits_{RUN_TAG}_s2.pt")
    write_submission(tl2, RUN_TAG + "_s2")


sentence vectors cached for 100 docs
stage2 epoch 1 | loss 0.1973 | dev CL 0.3848 IT 0.4723 ALL 0.4580
stage2 epoch 2 | loss 0.1926 | dev CL 0.3823 IT 0.4809 ALL 0.4591
stage2 epoch 3 | loss 0.1882 | dev CL 0.3840 IT 0.4781 ALL 0.4584
stage2 epoch 4 | loss 0.1837 | dev CL 0.3878 IT 0.4857 ALL 0.4625
stage2 epoch 5 | loss 0.1807 | dev CL 0.3862 IT 0.4781 ALL 0.4590
stage2 epoch 6 | loss 0.1750 | dev CL 0.3867 IT 0.4789 ALL 0.4610
stage2 epoch 7 | loss 0.1713 | dev CL 0.3903 IT 0.4753 ALL 0.4608
stage2 epoch 8 | loss 0.1676 | dev CL 0.3895 IT 0.4825 ALL 0.4627
stage2 epoch 9 | loss 0.1639 | dev CL 0.3904 IT 0.4794 ALL 0.4626
stage2 epoch 10 | loss 0.1604 | dev CL 0.3875 IT 0.4780 ALL 0.4596
stage2 epoch 11 | loss 0.1560 | dev CL 0.3890 IT 0.4779 ALL 0.4598
stage2 epoch 12 | loss 0.1523 | dev CL 0.3888 IT 0.4730 ALL 0.4583
stage2 epoch 13 | loss 0.1489 | dev CL 0.3897 IT 0.4803 ALL 0.4614
stage2 epoch 14 | loss 0.1447 | dev CL 0.3862 IT 0.4812 ALL 0.4593
stage2 epoch 15 | loss 0.1411 | de

### 3.6b Stage-2 comparison table
List (loss, CLASS_W_POWER) combos; one run prints the official dev score for each and saves their logits for 3.7 / 3.8.

In [157]:
# ===== 3.6b Stage-2 comparison table (plumbing): one run tries every combo YOU list, prints the official dev score =====
# Needs 3.5 and 3.6 run first in this session (uses their cached sentence vectors). Each combo restarts the doc layer
# from the same post-3.5 weights, so results are comparable. Nothing is chosen automatically: you read the table and pick.
SWEEP = [("ce", 0.2), ("focal", 0.2)]   # YOUR combos as (loss, CLASS_W_POWER), e.g. [("ce", 0.2), ("focal", 0.2)]
if PHASE == 3 and SWEEP:
    s1 = torch.load(f"/kaggle/working/weights_{RUN_TAG}.pt")
    doc_s1 = {k: v for k, v in s1.items() if k.startswith(("doc_lstm.", "head."))}
    loss_fns = {"ce": weighted_ce, "focal": focal_loss}
    table = []
    for loss_name, cwp in SWEEP:
        if loss_name == "focal": need("FOCAL_GAMMA")
        model.load_state_dict(doc_s1, strict=False)
        for p in model.parameters(): p.requires_grad = False
        dps = list(model.doc_lstm.parameters()) + list(model.head.parameters())
        for p in dps: p.requires_grad = True
        opt_s = torch.optim.AdamW(dps, lr=HP["STAGE2_LR"], weight_decay=HP["WD"])
        cw_s = class_weights_rr(s2_train, cwp)
        best_s, state_s = -1.0, None
        for ep in range(HP["STAGE2_EPOCHS"]):
            random.shuffle(s2_train); model.train()
            for it in s2_train:
                lg = doc_logits(model, SV[it["id"]])
                loss = loss_fns[loss_name](lg.float(), torch.tensor(it["labels"], device=device), cw_s)
                opt_s.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(dps, 1.0); opt_s.step()
            f = evaluate(DEV, s2_predict(DEV))["ALL"]
            if f > best_s:
                best_s, state_s = f, {n: q.detach().cpu().clone() for n, q in model.named_parameters() if q.requires_grad}
        model.load_state_dict(state_s, strict=False)
        tag = f"{RUN_TAG}_s2_{loss_name}_cw{cwp}"
        torch.save(s2_predict(DEV),  f"/kaggle/working/devlogits_{tag}.pt")
        torch.save(s2_predict(TEST), f"/kaggle/working/testlogits_{tag}.pt")
        table.append((loss_name, cwp, best_s))
        print(f"{loss_name:6s} CLASS_W_POWER {cwp}: best dev macro-F1 (official) {best_s:.4f}  -> devlogits/testlogits_{tag}.pt")
    print("\nsummary:"); [print(f"  {l:6s} cw {c_}: {f:.4f}") for l, c_, f in table]
    # to submit one: put its testlogits/devlogits file names into 3.7 (single file is fine) or 3.8 (with TAU)


ce     CLASS_W_POWER 0.2: best dev macro-F1 (official) 0.4533  -> devlogits/testlogits_p3_arch3_lora_dev_s2_ce_cw0.2.pt
focal  CLASS_W_POWER 0.2: best dev macro-F1 (official) 0.4627  -> devlogits/testlogits_p3_arch3_lora_dev_s2_focal_cw0.2.pt

summary:
  ce     cw 0.2: 0.4533
  focal  cw 0.2: 0.4627


### 3.7 Ensemble
List `testlogits_*.pt` (and optionally `devlogits_*.pt`) files from your runs; writes `submission_p3_ensemble.zip`.

In [158]:
# ===== 3.7 Ensemble: average probabilities of several runs (plumbing, equal weights) =====
ENS_TEST = []        # e.g. ["testlogits_p3_arch3_lora_dev_s2.pt", "testlogits_p3_arch2_lorma_dev_s2.pt"]
ENS_DEV  = []        # matching devlogits_*.pt files (optional: scores the ensemble on dev)

def _load_logits(name):
    for p in (name, f"/kaggle/working/{name}"):
        if os.path.exists(p): return torch.load(p)
    return torch.load(find_input(name))

def _avg_probs(names):
    acc = None
    for n in names:
        P_ = {k: torch.softmax(v.float(), -1) for k, v in _load_logits(n).items()}
        acc = P_ if acc is None else {k: acc[k] + P_[k] for k in acc}
    return {k: v / len(names) for k, v in acc.items()}

if PHASE == 3 and ENS_DEV:
    print("ENSEMBLE DEV:", evaluate(DEV, _avg_probs(ENS_DEV), show_per_class=True))
if PHASE == 3 and ENS_TEST:
    write_submission(_avg_probs(ENS_TEST), "p3_ensemble")


### 3.8 Logit adjustment (post-processing for rare labels)
Runs on saved logits in seconds; compare TAU values on dev, then set TAU to write a submission.

In [159]:
# ===== 3.8 Logit adjustment (post-processing; Menon et al. 2021): score_c = log p_c - TAU * log prior_c =====
# Works on saved logits: one file = single model, several files = ensemble average (same as 3.7).
# prior_c = label frequency in TRAIN. TAU = 0 changes nothing; TAU > 0 shifts predictions toward rarer labels.
LA_DEV    = []      # e.g. ["devlogits_p3_arch3_lora_dev_s2.pt"]
LA_TEST   = []      # matching testlogits files, same order
TAU_SWEEP = []      # YOUR candidate values to compare on dev (official metric)
TAU       = None    # YOUR choice after looking at the sweep -> writes the submission

if PHASE == 3 and LA_DEV:
    cnt = torch.zeros(len(LABELS))
    for it in TRAIN:
        cnt += torch.bincount(torch.tensor(it["labels"]), minlength=len(LABELS)).float()
    log_prior = torch.log((cnt + 1) / (cnt + 1).sum())

    def adjust(probs_by_id, tau):
        return {k: torch.log(v.clamp_min(1e-12)) - tau * log_prior for k, v in probs_by_id.items()}

    dev_p = _avg_probs(LA_DEV)
    print("no adjustment:", evaluate(DEV, adjust(dev_p, 0.0)))
    for tau in TAU_SWEEP:
        print(f"tau {tau}: dev", evaluate(DEV, adjust(dev_p, tau)))
    if TAU is not None and LA_TEST:
        print(f"chosen tau {TAU}: dev", evaluate(DEV, adjust(dev_p, TAU), show_per_class=True))
        write_submission(adjust(_avg_probs(LA_TEST), TAU), f"p3_la_tau{TAU}")


# PHASE 4 — Court Judgment Prediction
Run order: 4.1 → 4.5. Per account, edit only the CONTROL cell (`ARCH`, `TECHNIQUE`, `BIDIR`, `KEEP`, `TIME_BUDGET_H`, `EPOCHS`, `CACHE_TEACHER_LOGITS`, `TEACHER_TAG`).

### 4.1 Data + checks
Loads train/dev/test, reports duplicates, stops if any case is shared with test (leak check), prints label balance and lengths.

In [7]:
if PHASE == 4:
    import hashlib

    def read_json_any(path):
        try:
            return json.load(open(path))
        except json.JSONDecodeError:                                   # JSON-lines fallback
            return [json.loads(l) for l in open(path) if l.strip()]

    def to_records(obj, name):
        if isinstance(obj, dict):
            vals = list(obj.values())
            if vals and all(isinstance(v, dict) for v in vals):        # {case_id: {text, label, ...}}
                obj = [dict(id=k, **v) for k, v in obj.items()]
            elif any(isinstance(v, list) and v and isinstance(v[0], dict) for v in vals):
                obj = next(v for v in vals if isinstance(v, list) and v and isinstance(v[0], dict))  # {"data": [...]}
            elif "id" in obj and isinstance(obj["id"], list):          # columnar {id: [...], text: [...], label: [...]}
                n = len(obj["id"])
                obj = [{k: obj[k][i] for k in obj if isinstance(obj[k], list) and len(obj[k]) == n} for i in range(n)]
            else:
                raise ValueError(f"{name}: unknown layout, top keys {list(obj)[:5]}")
        assert isinstance(obj, list) and obj and isinstance(obj[0], dict), f"{name}: expected a list of records"
        return obj

    def pick(keys, cands, what, required=True):
        for c in cands:
            if c in keys: return c
        if required: raise KeyError(f"no {what} field among {sorted(keys)}")
        return None

    def to_label(v):
        if isinstance(v, list): v = v[0]
        v = int(float(v))
        assert v in (0, 1), f"non-binary label {v}"
        return v

    # ---- load the three splits ----
    RAW4 = {}
    for split, key in (("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")):
        recs = to_records(read_json_any(P[key]), key)
        keys = set(recs[0].keys())
        ID_K   = pick(keys, ("id", "case_id", "doc_id", "name"), "id")
        TEXT_K = pick(keys, ("text", "case_text", "document", "judgment"), "text")
        LAB_K  = pick(keys, ("label", "labels", "outcome", "decision", "pred"), "label", required=(split != "test"))
        out = []
        for r in recs:
            t = r[TEXT_K]
            if isinstance(t, list): t = " ".join(map(str, t))           # sentence-list form -> one string
            d = {"id": r[ID_K], "text": t}                                # id kept with its ORIGINAL type for the writer
            if LAB_K and split != "test": d["label"] = to_label(r[LAB_K])
            out.append(d)
        RAW4[split] = out
        extra = sorted(keys - {ID_K, TEXT_K, LAB_K})
        print(f"{split:5s}: {len(out):6d} cases | id='{ID_K}' text='{TEXT_K}' label='{LAB_K}' | other fields (not used): {extra}")

    # ---- integrity checks: duplicates inside a split are reported; anything shared with test stops the cell ----
    norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
    md5  = lambda s: hashlib.md5(norm(s).encode()).hexdigest()
    IDS  = {s: [str(d["id"]) for d in RAW4[s]] for s in RAW4}
    HASH = {s: {md5(d["text"]) for d in RAW4[s]} for s in RAW4}
    for s in RAW4:
        dup = [k for k, n in collections.Counter(IDS[s]).items() if n > 1]
        print(f"{s:5s}: unique ids {len(set(IDS[s]))} / {len(IDS[s])} | duplicated ids {len(dup)} | unique texts {len(HASH[s])}")
        if dup:
            by = collections.defaultdict(list)
            for d in RAW4[s]: by[str(d["id"])].append(d)
            same_t = sum(len({md5(x["text"]) for x in by[k]}) == 1 for k in dup)
            same_l = sum(len({x.get("label") for x in by[k]}) == 1 for k in dup)
            print(f"   of the {len(dup)} duplicated ids: identical text {same_t} | same label {same_l}")
    assert len(IDS["test"]) == len(set(IDS["test"])), "duplicate ids inside test -- one prediction per id would be ambiguous"
    for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
        ni, nt = len(set(IDS[a]) & set(IDS[b])), len(HASH[a] & HASH[b])
        print(f"overlap {a}-{b}: ids {ni} | identical texts {nt}")
        if b == "test":
            assert ni == 0 and nt == 0, f"{a} shares cases with test -- old (leaked) data? re-download"

    # ---- label balance + length statistics ----
    def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))]
    for s in ("train", "dev"):
        c = collections.Counter(d["label"] for d in RAW4[s])
        print(f"{s:5s} labels: 0 -> {c[0]} ({c[0]/len(RAW4[s]):.1%}) | 1 -> {c[1]} ({c[1]/len(RAW4[s]):.1%})")
    for s in RAW4:
        wl = [len(d["text"].split()) for d in RAW4[s]]
        print(f"{s:5s} words/case p50 {pct(wl,.5)} p90 {pct(wl,.9)} p99 {pct(wl,.99)} max {max(wl)} | total {sum(wl)/1e6:.1f}M words")

    # ---- tokenizer cost on a sample (your BPE is pure Python, so full-text encoding of every case is the slow step) ----
    rng = random.Random(SEED)
    sample = rng.sample(RAW4["train"], min(100, len(RAW4["train"])))
    t0 = time.time()
    lens = [len(tok.encode(d["text"])) for d in sample]
    dt = time.time() - t0
    ntok = sum(lens)
    nw = sum(len(d["text"].split()) for d in sample)
    tpw = ntok / nw
    total_w = sum(len(d["text"].split()) for s in RAW4 for d in RAW4[s])
    print(f"sample of {len(sample)}: {tpw:.2f} tokens/word | {ntok/dt:,.0f} tokens/s encode speed")
    print(f"tokens/case (sample): p50 {pct(lens,.5)} p90 {pct(lens,.9)} max {max(lens)}")
    print(f"projected full-text tokens: {total_w*tpw/1e6:.0f}M | projected encode time for ALL splits: {total_w*tpw/(ntok/dt)/3600:.1f} h")

    ex = RAW4["train"][0]

train:  37387 cases | id='id' text='text' label='label' | other fields (not used): ['expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
dev  :   1988 cases | id='id' text='text' label='label' | other fields (not used): ['expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
test :   1517 cases | id='id' text='text' label='label' | other fields (not used): ['expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train: unique ids 32305 / 37387 | duplicated ids 5082 | unique texts 31652
   of the 5082 duplicated ids: identical text 5082 | same label 5082
dev  : unique ids 994 / 1988 | duplicated ids 994 | unique texts 994
   of the 994 duplicated ids: identical text 994 | same label 994
test : unique ids 1517 / 1517 | duplicated ids 0 | unique texts 1517
overlap train-dev: ids 0 | identical texts 0
overlap train-test: ids 0 | identical texts 0
overlap dev-test: ids 0 | identical texts 0
train labels: 0 -> 22067 (59.0%) | 1 -> 15320 (41.0%)
dev   labels: 0 -> 994 (50.0%) 

### 4.2 De-duplicate, tokenize once (cached), chunking
Your settings: `DEDUP`, `CHUNK_LEN`, `OVERLAP`, `MAX_CHUNKS`, `KEEP`. Prints coverage and hours per epoch.

In [8]:
if PHASE == 4:
    # ---- 4.2  tokenize every case ONCE, cache to a file, and define the chunking (plumbing) ----
    # Your choices (CONTROL / HP): CHUNK_LEN, OVERLAP, MAX_CHUNKS, KEEP, DEDUP.  The cache stores FULL token
    # sequences, so changing MAX_CHUNKS / KEEP later needs no re-tokenization; changing CHUNK_LEN/OVERLAP doesn't either.
    CACHE_NAME = "p4_tokens.pt"

    # 1) de-duplicate (by id; all repeated ids were verified identical in text and label in 4.1)
    def dedup(recs):
        seen, out = set(), []
        for d in recs:
            k = str(d["id"])
            if k not in seen: seen.add(k); out.append(d)
        return out
    DATA4 = {s: (dedup(RAW4[s]) if HP["DEDUP"] else list(RAW4[s])) for s in RAW4}
    for s in DATA4: print(f"{s:5s}: {len(RAW4[s])} rows -> {len(DATA4[s])} cases")
    by_text = collections.defaultdict(list)                       # report only: same text under different ids
    for d in DATA4["train"]: by_text[md5(d["text"])].append(d["label"])
    shared = [v for v in by_text.values() if len(v) > 1]
    print(f"train texts shared by >1 id: {len(shared)} | of these with conflicting labels: {sum(len(set(v)) > 1 for v in shared)}  (kept as-is)")

    # 2) tokenize once (or load the cache if it is attached from another account)
    hits = glob.glob(f"/kaggle/input/**/{CACHE_NAME}", recursive=True) + glob.glob(f"/kaggle/working/{CACHE_NAME}")
    TOK4 = None
    if hits:
        TOK4 = torch.load(hits[0], weights_only=False)
        if TOK4["cfg"]["DEDUP"] != HP["DEDUP"] or any(TOK4["ids"][s] != [str(d["id"]) for d in DATA4[s]] for s in DATA4):
            print("cache does not match the current data/DEDUP setting -> re-tokenizing"); TOK4 = None
        else:
            print("loaded token cache:", hits[0])
    if TOK4 is None:
        TOK4 = {"cfg": {"DEDUP": HP["DEDUP"]}, "ids": {}, "labels": {}, "offsets": {}, "tokens": {}}
        for s in DATA4:
            t0 = time.time(); seqs = []
            for i, d in enumerate(DATA4[s]):
                seqs.append(np.asarray(tok.encode(d["text"]), dtype=np.uint16))     # vocab 65,355 < 65,536
                if i % 5000 == 0: print(f"  {s}: {i}/{len(DATA4[s])}  {time.time()-t0:.0f}s")
            TOK4["ids"][s]     = [str(d["id"]) for d in DATA4[s]]
            TOK4["labels"][s]  = [d.get("label", -1) for d in DATA4[s]]
            TOK4["offsets"][s] = np.concatenate([[0], np.cumsum([len(x) for x in seqs])]).astype(np.int64)
            TOK4["tokens"][s]  = np.concatenate(seqs)
            print(f"{s:5s}: {len(seqs)} cases, {len(TOK4['tokens'][s])/1e6:.1f}M tokens, {time.time()-t0:.0f}s")
        torch.save(TOK4, f"/kaggle/working/{CACHE_NAME}")
        print(f"saved /kaggle/working/{CACHE_NAME} ({os.path.getsize(f'/kaggle/working/{CACHE_NAME}')/1e6:.0f} MB)"
              " -> make it a dataset and attach it on the other accounts to skip this step")
    ORIG_ID = {s: {str(d["id"]): d["id"] for d in DATA4[s]} for s in DATA4}   # original id type, for the writer

    # 3) chunking
    def doc_tokens(split, i):
        o = TOK4["offsets"][split]
        return TOK4["tokens"][split][o[i]:o[i + 1]]

    def chunk_starts(n, L, stride):
        """start offsets of overlapping windows covering all n tokens; the last window is aligned to the end"""
        if n <= L: return [0]
        st = list(range(0, n - L + 1, stride))
        if st[-1] + L < n: st.append(n - L)
        return st

    def select_chunks(st, max_chunks, keep):
        if len(st) <= max_chunks: return st
        if keep == "first": return st[:max_chunks]
        if keep == "last":  return st[-max_chunks:]
        if keep == "both":
            a = (max_chunks + 1) // 2
            return st[:a] + st[len(st) - (max_chunks - a):]
        raise ValueError("HP['KEEP'] must be 'first', 'last' or 'both'")

    def doc_chunks(split, i):
        """list of int64 tensors (each <= CHUNK_LEN tokens) for case i of a split, after the cap"""
        x = doc_tokens(split, i)
        L = HP["CHUNK_LEN"]; stride = L - int(round(L * HP["OVERLAP"]))
        st = select_chunks(chunk_starts(len(x), L, stride), HP["MAX_CHUNKS"], HP["KEEP"])
        return [torch.from_numpy(x[a:a + L].astype(np.int64)) for a in st]

    # 4) what the settings imply: coverage, tokens per epoch, time at your measured T4 speed (~5k tok/s, Phase-1 arch 1)
    L = HP["CHUNK_LEN"]; stride = L - int(round(L * HP["OVERLAP"]))
    print(f"\nCHUNK_LEN {L} | overlap {L - stride} tokens | stride {stride} | MAX_CHUNKS {HP['MAX_CHUNKS']} "
          f"-> a case is fully covered up to {L + (HP['MAX_CHUNKS'] - 1) * stride} tokens")
    for s in DATA4:
        o = TOK4["offsets"][s]; n = np.diff(o)
        nch = np.array([len(chunk_starts(int(k), L, stride)) for k in n])
        cap = np.minimum(nch, HP["MAX_CHUNKS"])
        used = sum(min(int(k), L) if c == 1 else c * L for k, c in zip(n, cap))
        print(f"{s:5s}: chunks/case p50 {int(np.median(nch))} p90 {int(np.percentile(nch, 90))} max {nch.max()} | "
              f"cases over the cap {np.mean(nch > HP['MAX_CHUNKS']):.1%} | tokens through the encoder {used/1e6:.1f}M")
        if s == "train": train_used = used
        else: globals()[f"used_{s}"] = used
    SPEED = 5000
    print(f"\ntrain epoch, fine-tuning (fwd+bwd) at ~{SPEED} tok/s: {train_used/SPEED/3600:.1f} h for all train cases "
          f"(scales linearly with the fraction of train cases used)")
    print(f"dev+test prediction (forward only, roughly 3x faster -- an assumption): {(used_dev + used_test)/(3*SPEED)/3600:.1f} h")
    print(f"frozen-encoder caching of train+dev+test (forward only): {(train_used + used_dev + used_test)/(3*SPEED)/3600:.1f} h")

train: 37387 rows -> 32305 cases
dev  : 1988 rows -> 994 cases
test : 1517 rows -> 1517 cases
train texts shared by >1 id: 642 | of these with conflicting labels: 0  (kept as-is)
  train: 0/32305  0s
  train: 5000/32305  16s
  train: 10000/32305  25s
  train: 15000/32305  35s
  train: 20000/32305  48s
  train: 25000/32305  59s
  train: 30000/32305  68s
train: 32305 cases, 122.1M tokens, 72s
  dev: 0/994  0s
dev  : 994 cases, 4.8M tokens, 3s
  test: 0/1517  0s
test : 1517 cases, 7.6M tokens, 5s
saved /kaggle/working/p4_tokens.pt (318 MB) -> make it a dataset and attach it on the other accounts to skip this step

CHUNK_LEN 1024 | overlap 205 tokens | stride 819 | MAX_CHUNKS 8 -> a case is fully covered up to 6757 tokens
train: chunks/case p50 3 p90 10 max 588 | cases over the cap 12.7% | tokens through the encoder 122.3M
dev  : chunks/case p50 5 p90 11 max 69 | cases over the cap 19.4% | tokens through the encoder 5.1M
test : chunks/case p50 5 p90 12 max 67 | cases over the cap 20.6% | t

### 4.3 YOUR CODE — the Phase-4 model
**Contract the harness relies on:** backbone stored as `self.encoder`; `__init__(encoder, d_model)`; `forward(win_ids, win_lens)` with `win_ids` `(n_chunks, L)` padded with EOS and `win_lens` `(n_chunks,)`; returns logits `(1, 2)`; token states via `encoder_states(self.encoder, win_ids, win_lens)`.

In [ ]:
if PHASE == 4:
    class CJPEModel(nn.Module):
        def __init__(self, encoder, d_model):
            super().__init__()
            self.encoder = encoder
            self.doc_rnn = nn.GRU(d_model, d_model // 2, bidirectional=True, batch_first=True)
            self.classifier = nn.Linear(d_model, 2)

        def forward(self, win_ids, win_lens):
            H = encoder_states(self.encoder, win_ids, win_lens)
            
            mask = torch.arange(H.size(1), device=H.device).unsqueeze(0) < win_lens.unsqueeze(1)
            H = H * mask.unsqueeze(2).float()
            chunk_reps = H.sum(dim=1) / win_lens.unsqueeze(1).clamp(min=1).float()
            
            chunk_reps = chunk_reps.unsqueeze(0)
            doc_out, _ = self.doc_rnn(chunk_reps)
            doc_rep = doc_out.mean(dim=1)
            
            logits = self.classifier(doc_rep)
            return logits

    def build_cjpe_model():
        enc, d_model = load_backbone()
        return apply_technique(CJPEModel(enc, d_model)).to(device)

    def build_student4():
        enc, d_model = load_backbone()
        
        n_before = len(enc.blocks)
        enc.blocks = torch.nn.ModuleList(list(enc.blocks)[:HP["STUDENT_LAYERS"]])
        assert len(enc.blocks) == HP["STUDENT_LAYERS"] < n_before, "student was not truncated"
        print(f"KD student: kept {len(enc.blocks)} of {n_before} blocks")
            
        model = CJPEModel(enc, d_model)
        
        for param in model.parameters():
            param.requires_grad = True
            
        return model.to(device)

### 4.4 Harness (training, evaluation, submission)
Speed check after 50 cases, time-budget stop, best-dev weights saved on every improvement, OOM cases skipped and counted.

In [ ]:
if PHASE == 4:
    # ===================== 4.4  Harness: data -> tensors, training, evaluation, submission (plumbing) =====================
    from sklearn.metrics import f1_score, precision_score, recall_score

    def p4_items(split, frac=None, seed=0):
        """indices of the cases of a split (optionally a random fraction, train only)"""
        idx = list(range(len(TOK4["ids"][split])))
        if frac is not None and frac < 1.0:
            idx = sorted(random.Random(seed).sample(idx, int(round(frac * len(idx)))))
        return idx

    def p4_tensors(split, i):
        ch = doc_chunks(split, i)
        L = max(len(c) for c in ch)
        x = torch.full((len(ch), L), EOS_ID, dtype=torch.long)
        for k, c in enumerate(ch): x[k, :len(c)] = c
        return x.to(device), torch.tensor([len(c) for c in ch]).to(device)

    def p4_label(split, i): return int(TOK4["labels"][split][i])
    def p4_id(split, i):    return TOK4["ids"][split][i]

    def class_weights4(split, idx, power):
        cnt = torch.bincount(torch.tensor([p4_label(split, i) for i in idx]), minlength=2).float()
        w = (cnt.sum() / cnt.clamp(min=1)) ** power
        return (w / w.mean()).to(device)

    def metrics4(gold, pred):
        """Macro-F1 (primary) + macro precision / recall over the two classes."""
        return dict(f1=float(f1_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)),
                    p=float(precision_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)),
                    r=float(recall_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)))

    @torch.no_grad()
    def predict4(model, split, idx):
        model.eval(); out = {}
        for i in idx:
            x, lens = p4_tensors(split, i)
            with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                lg = model(x, lens)
            out[p4_id(split, i)] = lg.float().reshape(1, 2).cpu()
        model.train()
        return out                                          # case id -> (1, 2) logits

    def evaluate4(split, idx, logits_by_id):
        gold = [p4_label(split, i) for i in idx]
        pred = [int(logits_by_id[p4_id(split, i)].argmax(-1)) for i in idx]
        return metrics4(gold, pred)

    def trainable_state(model):
        return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

    def train4(model, train_idx, dev_idx, loss_fn, teacher=None, *, EPOCHS, LR_ENC, LR_HEAD, WARMUP_FRAC, WD,
               ACCUM_DOCS, CLASS_W_POWER, TIME_BUDGET_H, EVAL_EVERY=None, CLIP=1.0, seed=0, tag="run"):
        """One case per forward pass, gradient accumulation over ACCUM_DOCS cases.
        Stops cleanly when TIME_BUDGET_H hours of training have passed. Keeps the best-dev trainable weights."""
        torch.manual_seed(seed); rng = random.Random(seed)
        train_idx = list(train_idx)
        cw = class_weights4("train", train_idx, CLASS_W_POWER)
        enc_p  = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("encoder.")]
        head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("encoder.")]
        groups = [g for g in ({"params": enc_p, "lr": LR_ENC}, {"params": head_p, "lr": LR_HEAD}) if g["params"]]
        opt = torch.optim.AdamW(groups, weight_decay=WD)
        steps_ep = math.ceil(len(train_idx) / ACCUM_DOCS); total = EPOCHS * steps_ep
        warm = max(1, int(WARMUP_FRAC * total))
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
        scaler = torch.amp.GradScaler(DEV_TYPE, enabled=(DEV_TYPE == "cuda"))
        params = [p for g in groups for p in g["params"]]
        best = {"f1": -1.0}; best_state = None; hist = []; n_oom = 0
        t_start = time.time(); stop = False; seen_tok = 0; done = 0
        _probe = random.Random(seed + 1).sample(train_idx, min(200, len(train_idx)))       # random sample, not file order
        n_tok_epoch = sum(int(sum(len(c) for c in doc_chunks("train", i))) for i in _probe) / len(_probe) * len(train_idx)
        win_l, win_n, win_p1, win_y1 = 0.0, 0, 0, 0                                        # recent-window diagnostics

        def dev_check(label):
            nonlocal best, best_state
            if not dev_idx: return
            res = evaluate4("dev", dev_idx, predict4(model, "dev", dev_idx)); res["at"] = label; hist.append(res)
            print(f"  [{label}] dev macro-F1 {res['f1']:.4f} | P {res['p']:.4f} | R {res['r']:.4f}")
            if res["f1"] > best["f1"]:
                best, best_state = res, trainable_state(model)
                torch.save(best_state, f"/kaggle/working/weights_{tag}.pt")       # survives a session kill

        model.train()
        for ep in range(EPOCHS):
            rng.shuffle(train_idx); t0 = time.time(); tot = 0.0
            for k, i in enumerate(train_idx):
                x, lens = p4_tensors("train", i)
                y = torch.tensor([p4_label("train", i)], device=device)
                tch = teacher[p4_id("train", i)].to(device) if teacher is not None else None
                try:
                    with torch.autocast(DEV_TYPE, dtype=torch.float16, enabled=(DEV_TYPE == "cuda")):
                        lg = model(x, lens)
                    loss = loss_fn(lg.float().reshape(1, 2), y, cw, teacher=tch) / ACCUM_DOCS
                    scaler.scale(loss).backward(); tot += loss.item() * ACCUM_DOCS
                    win_l += loss.item() * ACCUM_DOCS; win_n += 1
                    win_p1 += int(lg.reshape(1, 2).argmax(-1).item() == 1); win_y1 += int(y.item() == 1)
                except torch.cuda.OutOfMemoryError:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    print(f"  OOM on case {p4_id('train', i)} ({x.shape[0]} chunks) -- skipped ({n_oom} so far)")
                    continue
                seen_tok += int(lens.sum()); done += 1
                if (k + 1) % ACCUM_DOCS == 0 or k + 1 == len(train_idx):
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, CLIP)
                    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                el = time.time() - t_start
                if done == 50:                                                     # early speed check
                    sp = seen_tok / el
                    print(f"  speed check: {sp:,.0f} tokens/s | projected epoch {n_tok_epoch / sp / 3600:.2f} h "
                          f"| budget {TIME_BUDGET_H} h | GPU mem peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"
                          if DEV_TYPE == "cuda" else f"  speed check: {sp:,.0f} tokens/s")
                    steps_fit = int(TIME_BUDGET_H * 3600 * done / el / ACCUM_DOCS)
                    print(f"  LR schedule: {total} planned steps (EPOCHS={EPOCHS}), warmup {warm} | about {steps_fit} steps fit in the budget")
                    if steps_fit < warm:
                        print("  !! the budget ends INSIDE warmup: LR never reaches LR_ENC/LR_HEAD -- check EPOCHS / WARMUP_FRAC vs TIME_BUDGET_H")
                    elif steps_fit < total:
                        print("  !! the budget ends before the planned schedule finishes: LR will not decay to 0 -- check EPOCHS vs TIME_BUDGET_H")
                if (k + 1) % 1000 == 0 and win_n:
                    print(f"  ep {ep + 1} | {k + 1}/{len(train_idx)} cases | last-{win_n} loss {win_l / win_n:.4f} "
                          f"(50/50 guess = 0.693) | predicted 1: {win_p1 / win_n:.0%} vs true 1: {win_y1 / win_n:.0%} "
                          f"| LR {sched.get_last_lr()[0]:.2e} | {el / 3600:.2f} h")
                    win_l, win_n, win_p1, win_y1 = 0.0, 0, 0, 0
                if EVAL_EVERY and (k + 1) % EVAL_EVERY == 0:
                    dev_check(f"ep{ep + 1} case{k + 1}")
                if el > TIME_BUDGET_H * 3600:
                    print(f"  time budget reached after {k + 1} cases of epoch {ep + 1}"); stop = True; break
            print(f"epoch {ep + 1}/{EPOCHS} | train loss {tot / max(1, k + 1):.4f} | {time.time() - t0:.0f}s | OOM-skipped {n_oom}")
            dev_check(f"end ep{ep + 1}" if not stop else f"stop ep{ep + 1}")
            if stop: break
        if best_state is not None:
            model.load_state_dict(best_state, strict=False)
            print("BEST dev:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in best.items()})
        return best, hist

    def write_submission4(logits_by_id, tag):
        pred = []
        for cid in TOK4["ids"]["test"]:
            pred.append({"id": ORIG_ID["test"][cid], "pred": int(logits_by_id[cid].argmax(-1))})
        want = [str(d["id"]) for d in RAW4["test"]]
        assert len(pred) == len(set(want)) == len(want) and {str(p["id"]) for p in pred} == set(want), "test id mismatch"
        assert all(p["pred"] in (0, 1) for p in pred)
        js = f"/kaggle/working/prediction.json"; zp = f"/kaggle/working/submission_{tag}.zip"
        json.dump(pred, open(js, "w"))
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(js, arcname="prediction.json")
        c = collections.Counter(p["pred"] for p in pred)
        print(f"wrote {zp}: {len(pred)} predictions | pred 0: {c[0]}  pred 1: {c[1]}")

### 4.5 Run (uses the CONTROL cell)
Writes `submission_p4_<arch>_<technique>.zip`, dev/test logits, history; teacher logits when `CACHE_TEACHER_LOGITS = True`.

In [ ]:
if PHASE == 4:
    # ===================== 4.5  Run (uses the CONTROL cell) =====================
    P4_TECH = ("full", "partial", "freezing", "bitfit", "lorma", "adalora", "adapters", "kd")
    assert TECHNIQUE in P4_TECH, f"TECHNIQUE={TECHNIQUE!r} is not on the Phase-4 list {P4_TECH}"
    assert TECHNIQUE in ("full", "lorma", "kd") or TECHNIQUE in EXTRA_TECHNIQUES, f"write '{TECHNIQUE}' into EXTRA_TECHNIQUES first"
    need("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS", "CLASS_W_POWER", "TIME_BUDGET_H", "KEEP")
    tag4 = f"p4_{ARCH}_{TECHNIQUE}"
    settings = {k: HP[k] for k in ("EPOCHS", "LR_ENC", "LR_HEAD", "WARMUP_FRAC", "WD", "ACCUM_DOCS",
                                   "CLASS_W_POWER", "TIME_BUDGET_H")}
    settings["EVAL_EVERY"] = HP.get("EVAL_EVERY")
    train_idx = p4_items("train", HP.get("TRAIN_FRAC"), SEED)
    dev_idx, test_idx = p4_items("dev"), p4_items("test")

    teacher, loss_fn = None, train_loss()
    if TECHNIQUE == "kd":
        assert TEACHER_TAG, "set TEACHER_TAG in the CONTROL cell (e.g. 'arch3_full') and attach that run's teacher file"
        teacher = torch.load(find_input(f"teacher_logits_p4_{TEACHER_TAG}.pt"))
        train_idx = [i for i in train_idx if p4_id("train", i) in teacher]          # cases the teacher labelled
        model, loss_fn = build_student4().to(device), kd_loss
        trainable_report(model, "KD student")
    else:
        model = build_cjpe_model()
    print(f"{tag4}: train cases {len(train_idx)} | dev {len(dev_idx)} | test {len(test_idx)} | KEEP={HP['KEEP']}")

    t_start = time.time()
    best, hist = train4(model, train_idx, dev_idx, loss_fn, teacher, seed=SEED, tag=tag4, **settings)
    print(f"training time {(time.time() - t_start) / 3600:.2f} h")

    dev_logits = predict4(model, "dev", dev_idx)
    print("DEV (best weights):", evaluate4("dev", dev_idx, dev_logits))
    torch.save(dev_logits, f"/kaggle/working/devlogits_{tag4}.pt")
    test_logits = predict4(model, "test", test_idx)
    torch.save(test_logits, f"/kaggle/working/testlogits_{tag4}.pt")
    write_submission4(test_logits, tag4)
    json.dump(dict(hist=hist, settings=settings, hp={k: v for k, v in HP.items() if not callable(v)}),
              open(f"/kaggle/working/hist_{tag4}.json", "w"), default=str)

    if CACHE_TEACHER_LOGITS:                                    # this run will serve as the KD teacher
        t0 = time.time()
        tl = predict4(model, "train", train_idx)
        torch.save(tl, f"/kaggle/working/teacher_logits_p4_{ARCH}_{TECHNIQUE}.pt")
        print(f"saved teacher logits for {len(tl)} train cases in {(time.time() - t0) / 3600:.2f} h")

# PHASE 5 — Legal Statute Identification
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._

# PHASE 6 — Abstractive Summarization (uses the causal 'lm' checkpoints)
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._

# PHASE 7 — Prior Case Retrieval
_Section added when the phase opens. Data files are already listed in `PHASE_FILES`._